In [1]:
import pandas as pd 
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# 1
import requests
from io import BytesIO

# 1:    Data Organisation

In [2]:
def get_nifty500():
    """
    Download current NIFTY 500 constituents from NSE
    and return a normalized DataFrame.
    """

    url = "https://nsearchives.nseindia.com/content/indices/ind_nifty500list.csv"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/151.0.0.0 Safari/537.36"
        ),
        "Accept": "text/csv,application/csv,*/*",
        "Referer": "https://www.nseindia.com/"
    }

    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(BytesIO(response.content))

    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("&", "and")
    )

    # Normalize string columns
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()

    # Normalize symbol
    if "symbol" in df.columns:
        df["symbol"] = df["symbol"].str.upper()

    # Remove duplicate symbols
    df = df.drop_duplicates(subset="symbol")

    # Sort alphabetically
    df = df.sort_values("symbol").reset_index(drop=True)

    return df

In [3]:
# Fetching Nifty500 constisuents
nifty500 = get_nifty500()

print(nifty500.shape)
print(nifty500.head())

(501, 5)
                  company_name            industry     symbol series  \
0             360 ONE WAM Ltd.  Financial Services     360ONE     EQ   
1                3M India Ltd.         Diversified    3MINDIA     EQ   
2  Aadhar Housing Finance Ltd.  Financial Services  AADHARHFC     EQ   
3        Aarti Industries Ltd.           Chemicals   AARTIIND     EQ   
4        Aavas Financiers Ltd.  Financial Services      AAVAS     EQ   

      isin_code  
0  INE466L01038  
1  INE470A01017  
2  INE883F01010  
3  INE769A01020  
4  INE216P01012  


In [4]:
universe= nifty500["symbol"].to_list()
universe[:5]

['360ONE', '3MINDIA', 'AADHARHFC', 'AARTIIND', 'AAVAS']

In [5]:
# Fetching price data from yfinance

tickers= [f"{tic}.NS" for tic in universe]
df = yf.download(tickers, auto_adjust= True, period= "1y", group_by= "column", threads= True)
df= df.stack(level= "Ticker", future_stack= True).reset_index()
df= df.drop(columns= ["Adj Close"])
df= df.rename(columns= {"Ticker": "symbol"})
df["symbol"]= df["symbol"].str.replace(".NS", "", regex= False)         #.str.upper().str.strip() ADDITIONAL CHECK
df["Date"]= pd.to_datetime(df["Date"])                                  # Normalise date columns
df.columns= [col.lower().replace(" ", "_") for col in df.columns]        # Clean standardised column names lower case and _
df= df.sort_values(["date", "symbol"]).reset_index(drop= True)

[*********************100%***********************]  501 of 501 completed

1 Failed download:
['DUMMYHEG.NS']: HTTPError('HTTP Error 404: ')


In [6]:
df

,date,symbol,close,high,low,open,volume
0,2025-09-08,360ONE,1028.765381,1043.009849,1022.434493,1030.644880,643249.0
1,2025-09-08,3MINDIA,30394.812500,30518.028053,29872.378557,29872.378557,9815.0
2,2025-09-08,AADHARHFC,510.450012,515.900024,506.200012,510.600006,227750.0
3,2025-09-08,AARTIIND,389.450012,394.200012,376.399994,377.649994,1106620.0
4,2025-09-08,AAVAS,1593.300049,1608.300049,1577.199951,1590.000000,272140.0
...,...,...,...,...,...,...,...
126247,2026-09-08,ZENSARTECH,450.549988,454.899994,448.000000,454.049988,259702.0
126248,2026-09-08,ZENTEC,1825.000000,1864.800049,1804.099976,1808.000000,581105.0
126249,2026-09-08,ZFCVINDIA,2517.399902,2537.899902,2472.199951,2491.699951,77877.0
126250,2026-09-08,ZYDUSLIFE,1145.500000,1167.699951,1142.000000,1161.699951,2009087.0


We are going to label out target variables:

Y= 1;   if Max Return > 4% && Max Draw Down > -2%
y= 0;   else y = 0

In [7]:
# Building the Target variable
horizon= 9
tp= 0.10
sl= 0.07

g= df.groupby("symbol", sort= False)
for day in range(1, horizon +1):

    df[f"future_open_{day}"]= g["open"].shift(-day)
    df[f"future_high_{day}"]= g["high"].shift(-day)
    df[f"future_low_{day}"]= g["low"].shift(-day)
    df[f"future_close_{day}"]= g["close"].shift(-day)

# cacl how many future days are actually avilable
future_close_cols= [
    f"future_close_{day}"
    for day in range(1, horizon +1)
]
df["days_available"]= df[future_close_cols].notna().sum(axis=1)

# Entry Price 
entry= df["future_open_1"]                                                          #df["close"]
tp_price= entry *(1 +tp)

# calc max returns over horizon
high_cols= [f"future_high_{day}" for day in range(1, horizon+1)]
df["max_return_9d"]= (df[high_cols].max(axis= 1)) /entry -1

# calc path dependent drawdown
running_peak= entry.copy()
dd_columns= []

for day in range(1, horizon +1):
    high= df[f"future_high_{day}"]
    low= df[f"future_low_{day}"]
    running_peak= pd.concat([running_peak, high], axis= 1).max(axis= 1)
    dd= low /running_peak -1
    df[f"dd_{day}"]= dd
    dd_columns.append(f"dd_{day}")

df["max_dd_9d"]= df[dd_columns].min(axis= 1)

# Determine first TP/SL event
event= np.full(len(df), "NONE", dtype=object)
event_day= np.full(len(df), np.nan)
running_peak= entry.copy()
unresolved= np.ones(len(df), dtype=bool)

for day in range(1, horizon +1):
    high= df[f"future_high_{day}"]
    low= df[f"future_low_{day}"]
    running_peak= pd.concat([running_peak, high], axis= 1).max(axis= 1)
    tp_hit= high>= tp_price
    sl_hit= low<= running_peak *(1 -sl)
    active= unresolved
    both= active &tp_hit &sl_hit
    tp_only= active &tp_hit &~sl_hit
    sl_only= active &sl_hit &~tp_hit

    event[both]= "BOTH"
    event_day[both]= day

    event[tp_only]= "TP"
    event_day[tp_only]= day

    event[sl_only]= "SL"
    event_day[sl_only]= day

    resolved= both | tp_only | sl_only
    unresolved[resolved]= False

df["first_event"]= event
df["event_day"]= event_day

df["censored"]= (
    df["first_event"].eq("NONE") &
    (df["days_available"] <horizon)
)

# Converting event to model target
df["target"]= np.select(
    [
        df["first_event"].eq("TP"),
        df["first_event"].eq("SL"),
        df["censored"]
    ],
    [
        1,
        -1,
        np.nan
    ],
    default= 0
)


In [8]:
df["first_event"].value_counts()

first_event
NONE    62691
SL      51010
TP       8056
BOTH     4495
Name: count, dtype: int64

In [9]:
df[df["symbol"]== "HDFCBANK"]["first_event"].value_counts()#.iloc[8:20]

first_event
NONE    207
SL       43
TP        2
Name: count, dtype: int64

In [10]:
df.columns

Index(['date', 'symbol', 'close', 'high', 'low', 'open', 'volume',
       'future_open_1', 'future_high_1', 'future_low_1', 'future_close_1',
       'future_open_2', 'future_high_2', 'future_low_2', 'future_close_2',
       'future_open_3', 'future_high_3', 'future_low_3', 'future_close_3',
       'future_open_4', 'future_high_4', 'future_low_4', 'future_close_4',
       'future_open_5', 'future_high_5', 'future_low_5', 'future_close_5',
       'future_open_6', 'future_high_6', 'future_low_6', 'future_close_6',
       'future_open_7', 'future_high_7', 'future_low_7', 'future_close_7',
       'future_open_8', 'future_high_8', 'future_low_8', 'future_close_8',
       'future_open_9', 'future_high_9', 'future_low_9', 'future_close_9',
       'days_available', 'max_return_9d', 'dd_1', 'dd_2', 'dd_3', 'dd_4',
       'dd_5', 'dd_6', 'dd_7', 'dd_8', 'dd_9', 'max_dd_9d', 'first_event',
       'event_day', 'censored', 'target'],
      dtype='str')